# RAG Abstention: A Small Learning Experiment

This lab uses a fictional policy and a local phi3:mini model.

It explores answering from evidence, refusing unsupported questions,
and skipping generation when retrieval finds no passages.

A retrieved passage is not necessarily sufficient evidence.
This three-question pilot does not establish production reliability.

In [1]:
import sys

print("Python version:", sys.version)
print("Python location:", sys.executable)

Python version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Python location: C:\Users\Khaleel\enterprise-rag-reliability-lab\.venv\Scripts\python.exe


In [2]:
policy_text = """
FICTIONAL COMPANY POLICY — LAB USE ONLY

1. Employees may work remotely up to three days per week.
2. Remote work requires approval from the employee's manager.
3. Company laptops must use full-disk encryption.
4. Lost laptops must be reported to IT within one hour.
5. Employees must not upload confidential documents to unapproved AI services.
"""

print(policy_text)


FICTIONAL COMPANY POLICY — LAB USE ONLY

1. Employees may work remotely up to three days per week.
2. Remote work requires approval from the employee's manager.
3. Company laptops must use full-disk encryption.
4. Lost laptops must be reported to IT within one hour.
5. Employees must not upload confidential documents to unapproved AI services.



In [3]:
test_questions = [
    {
        "question": "How many days per week may employees work remotely?",
        "answerable": True,
        "expected_answer": "Up to three days per week."
    },
    {
        "question": "What is the annual work-from-home allowance?",
        "answerable": False,
        "expected_answer": "The policy does not specify an allowance."
    }
]

for item in test_questions:
    print("Question:", item["question"])
    print("Answerable from policy:", item["answerable"])
    print("Expected answer:", item["expected_answer"])
    print()

Question: How many days per week may employees work remotely?
Answerable from policy: True
Expected answer: Up to three days per week.

Question: What is the annual work-from-home allowance?
Answerable from policy: False
Expected answer: The policy does not specify an allowance.



In [4]:
import requests

try:
    response = requests.get("http://localhost:11434/api/tags", timeout=10)
    response.raise_for_status()

    for model in response.json().get("models", []):
        print(model["name"])

except requests.RequestException as error:
    print("Could not reach Ollama:", error)

tinyllama:latest
gemma:2b
phi3:mini
phi3:latest
qwen3.5:latest


In [5]:
import requests

try:
    response = requests.get("http://localhost:11434/api/tags", timeout=10)
    response.raise_for_status()

    for model in response.json().get("models", []):
        print(model["name"])

except requests.RequestException as error:
    print("Could not reach Ollama:", error)

tinyllama:latest
gemma:2b
phi3:mini
phi3:latest
qwen3.5:latest


In [6]:
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "phi3:mini",
        "prompt": (
            f"Use the following policy to answer the question.\n\n"
            f"{policy_text}\n\n"
            "Question: What is the annual work-from-home allowance?"
        ),
        "stream": False,
        "options": {"temperature": 0, "num_predict": 150}
    },
    timeout=180
)

response.raise_for_status()
print(response.json()["response"])

Based on the provided company policy, there is no specific mention of an "annual work-from-home allowance." The policy states that employees may work remotely up to three days per week, but it does not quantify this in terms of an allowance. Therefore, the question cannot be answered directly from the given policy. It is possible that the company has a separate policy or agreement regarding compensation for remote work, but this is not included in the provided text.


In [7]:
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "phi3:mini",
        "prompt": (
            f"Use the following policy to answer the question.\n\n"
            f"{policy_text}\n\n"
            "Question: How many days per week may employees work remotely?"
        ),
        "stream": False,
        "options": {"temperature": 0, "num_predict": 150}
    },
    timeout=180
)

response.raise_for_status()
print(response.json()["response"])

According to the company policy, employees may work remotely up to three days per week.


In [8]:
baseline_results = [
    {
        "question": test_questions[0]["question"],
        "answerable": True,
        "observation": "Correctly answered: up to three days per week."
    },
    {
        "question": test_questions[1]["question"],
        "answerable": False,
        "observation": (
            "Correctly identified that the allowance is unspecified, "
            "but speculated about a separate policy."
        )
    }
]

print("Recorded", len(baseline_results), "baseline observations.")

Recorded 2 baseline observations.


In [9]:
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "phi3:mini",
        "prompt": (
            "Answer only from the policy below. "
            "If the answer is absent, reply exactly: "
            "'The policy does not specify this.' "
            "Do not speculate or add explanations.\n\n"
            f"{policy_text}\n\n"
            "Question: What is the annual work-from-home allowance?"
        ),
        "stream": False,
        "options": {"temperature": 0, "num_predict": 150}
    },
    timeout=180
)

response.raise_for_status()
print(response.json()["response"])

The policy does not specify this.


In [10]:
prompt_only_result = {
    "question": test_questions[1]["question"],
    "model": "phi3:mini",
    "answer": response.json()["response"],
    "observation": "Abstained without extra speculation in this run."
}

print(prompt_only_result)

{'question': 'What is the annual work-from-home allowance?', 'model': 'phi3:mini', 'answer': 'The policy does not specify this.', 'observation': 'Abstained without extra speculation in this run.'}


In [11]:
policy_passages = [
    line.strip()
    for line in policy_text.splitlines()
    if line.strip() and line.strip()[0].isdigit()
]

for number, passage in enumerate(policy_passages, start=1):
    print(f"Passage {number}: {passage}")

Passage 1: 1. Employees may work remotely up to three days per week.
Passage 2: 2. Remote work requires approval from the employee's manager.
Passage 3: 3. Company laptops must use full-disk encryption.
Passage 4: 4. Lost laptops must be reported to IT within one hour.
Passage 5: 5. Employees must not upload confidential documents to unapproved AI services.


In [12]:
import re

def retrieve_passages(question):
    stop_words = {"the", "a", "an", "is", "to", "of", "how", "what", "may"}
    question_words = set(re.findall(r"\b[a-z]+\b", question.lower())) - stop_words

    matches = []
    for passage in policy_passages:
        passage_words = set(re.findall(r"\b[a-z]+\b", passage.lower()))
        score = len(question_words & passage_words)
        if score > 0:
            matches.append((score, passage))

    return sorted(matches, key=lambda item: item[0], reverse=True)

for score, passage in retrieve_passages(test_questions[0]["question"]):
    print(f"Matched words: {score} | {passage}")

Matched words: 6 | 1. Employees may work remotely up to three days per week.
Matched words: 1 | 2. Remote work requires approval from the employee's manager.
Matched words: 1 | 5. Employees must not upload confidential documents to unapproved AI services.


In [13]:
for score, passage in retrieve_passages(test_questions[1]["question"]):
    print(f"Matched words: {score} | {passage}")

Matched words: 2 | 2. Remote work requires approval from the employee's manager.
Matched words: 1 | 1. Employees may work remotely up to three days per week.


In [14]:
retrieval_observation = {
    "question": test_questions[1]["question"],
    "matches": retrieve_passages(test_questions[1]["question"]),
    "finding": "Related passages were retrieved, but the answer is absent."
}

print(retrieval_observation["finding"])

Related passages were retrieved, but the answer is absent.


In [15]:
question = test_questions[1]["question"]
matches = retrieve_passages(question)

retrieved_context = "\n".join(
    passage for score, passage in matches[:2]
)

print("Question:", question)
print("\nRetrieved context:")
print(retrieved_context)

Question: What is the annual work-from-home allowance?

Retrieved context:
2. Remote work requires approval from the employee's manager.
1. Employees may work remotely up to three days per week.


In [16]:
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "phi3:mini",
        "prompt": (
            "Answer only from the context below. "
            "If the answer is absent, reply exactly: "
            "'The context does not specify this.' "
            "Do not speculate or add explanations.\n\n"
            f"Context:\n{retrieved_context}\n\n"
            f"Question: {question}"
        ),
        "stream": False,
        "options": {"temperature": 0, "num_predict": 150}
    },
    timeout=180
)

response.raise_for_status()
rag_answer = response.json()["response"]
print(rag_answer)

The context does not specify this.


In [17]:
rag_result = {
    "question": question,
    "retrieved_context": retrieved_context,
    "model": "phi3:mini",
    "answer": rag_answer,
    "observation": "Correctly abstained in this run."
}

print(rag_result)

{'question': 'What is the annual work-from-home allowance?', 'retrieved_context': "2. Remote work requires approval from the employee's manager.\n1. Employees may work remotely up to three days per week.", 'model': 'phi3:mini', 'answer': 'The context does not specify this.', 'observation': 'Correctly abstained in this run.'}


In [18]:
unrelated_question = "What dental benefits are provided?"
unrelated_matches = retrieve_passages(unrelated_question)

print("Question:", unrelated_question)
print("Retrieved passages:", unrelated_matches)

Question: What dental benefits are provided?
Retrieved passages: []


In [19]:
if not unrelated_matches:
    answer = "No matching policy passages were found. I cannot answer."
    model_called = False
else:
    answer = "Passages found; model evaluation is needed."
    model_called = False  # This cell only checks retrieval.

print("Answer:", answer)
print("Model called:", model_called)

Answer: No matching policy passages were found. I cannot answer.
Model called: False


In [20]:
empty_retrieval_result = {
    "question": unrelated_question,
    "retrieved_passages": unrelated_matches,
    "answer": answer,
    "model_called": model_called
}

print("Empty-retrieval result recorded.")

Empty-retrieval result recorded.


In [21]:
import json
from pathlib import Path

output_file = Path("pilot_results.json")

output_file.write_text(
    json.dumps(
        {
            "retrieved_context_test": rag_result,
            "empty_retrieval_test": empty_retrieval_result
        },
        indent=2
    ),
    encoding="utf-8"
)

print("Saved to:", output_file.resolve())

Saved to: C:\Users\Khaleel\enterprise-rag-reliability-lab\pilot_results.json


In [22]:
saved_results = json.loads(
    output_file.read_text(encoding="utf-8")
)

for test_name, result in saved_results.items():
    print("Test:", test_name)
    print("Question:", result["question"])
    print("Answer:", result["answer"])
    print()

Test: retrieved_context_test
Question: What is the annual work-from-home allowance?
Answer: The context does not specify this.

Test: empty_retrieval_test
Question: What dental benefits are provided?
Answer: No matching policy passages were found. I cannot answer.



In [23]:
policy_file = Path("fictional_policy.txt")
policy_file.write_text(policy_text.strip() + "\n", encoding="utf-8")

print("Saved to:", policy_file.resolve())

Saved to: C:\Users\Khaleel\enterprise-rag-reliability-lab\fictional_policy.txt


In [24]:
questions_file = Path("test_questions.json")

questions_file.write_text(
    json.dumps(test_questions, indent=2),
    encoding="utf-8"
)

print("Saved to:", questions_file.resolve())

Saved to: C:\Users\Khaleel\enterprise-rag-reliability-lab\test_questions.json


In [25]:
dental_test = {
    "question": unrelated_question,
    "answerable": False,
    "expected_answer": "The policy does not specify dental benefits."
}

if not any(
    item["question"] == dental_test["question"]
    for item in test_questions
):
    test_questions.append(dental_test)

questions_file.write_text(
    json.dumps(test_questions, indent=2),
    encoding="utf-8"
)

print("Saved test questions:", len(test_questions))

Saved test questions: 3


In [26]:
model_response = requests.get(
    "http://localhost:11434/api/tags",
    timeout=10
)
model_response.raise_for_status()

model_info = next(
    model
    for model in model_response.json()["models"]
    if model["name"] == "phi3:mini"
)

Path("model_metadata.json").write_text(
    json.dumps(model_info, indent=2),
    encoding="utf-8"
)

print("Model:", model_info["name"])
print("Digest:", model_info["digest"])

Model: phi3:mini
Digest: 4f222292793889a9a40a020799cfd28d53f3e01af25d48e06c5e708610fc47e9


In [27]:
Path(".gitignore").write_text(
    ".venv/\n"
    ".ipynb_checkpoints/\n"
    "__pycache__/\n"
    ".env\n",
    encoding="utf-8"
)

print("Created .gitignore")

Created .gitignore


In [28]:
import subprocess

packages = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True
)

Path("requirements.txt").write_text(packages, encoding="utf-8")

print("Created requirements.txt")

Created requirements.txt


In [29]:
Path("README.md").write_text(
    "# Enterprise RAG Reliability Lab\n\n"
    "A small learning experiment using a fictional company policy "
    "and a local phi3:mini model through Ollama.\n\n"
    "## What it explores\n"
    "- Answering questions from policy text.\n"
    "- Abstaining when the answer is absent.\n"
    "- Retrieving passages with simple word matching.\n"
    "- Skipping model generation when retrieval returns no matches.\n\n"
    "## Limitations\n"
    "This pilot contains three test questions. Word overlap does not "
    "prove answer support. These examples do not establish production "
    "reliability or general model performance.\n",
    encoding="utf-8"
)

print("Created README.md")

Created README.md


In [31]:
from pathlib import Path

print("Notebook working folder:", Path.cwd())

Notebook working folder: C:\Users\Khaleel\enterprise-rag-reliability-lab


In [32]:
question = "Who must authorize telecommuting?"
matches = retrieve_passages(question)

for score, passage in matches:
    print(f"Matched words: {score} | {passage}")

Matched words: 1 | 3. Company laptops must use full-disk encryption.
Matched words: 1 | 4. Lost laptops must be reported to IT within one hour.
Matched words: 1 | 5. Employees must not upload confidential documents to unapproved AI services.


In [33]:
paraphrase_failure = {
    "question": question,
    "expected_passage": policy_passages[1],
    "retrieved_passages": [passage for score, passage in matches],
    "finding": "Word matching missed the answer-bearing passage."
}

Path("paraphrase_failure.json").write_text(
    json.dumps(paraphrase_failure, indent=2),
    encoding="utf-8"
)

print(paraphrase_failure["finding"])

Word matching missed the answer-bearing passage.


In [35]:
paraphrase_context = "\n".join(
    passage for score, passage in matches[:2]
)

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "phi3:mini",
        "prompt": (
            "Answer only from the context below. "
            "If the answer is absent, reply exactly: "
            "'The context does not specify this.' "
            "Do not speculate or add explanations.\n\n"
            f"Context:\n{paraphrase_context}\n\n"
            f"Question: {question}"
        ),
        "stream": False,
        "options": {"temperature": 0, "num_predict": 150}
    },
    timeout=180
)

response.raise_for_status()
paraphrase_answer = response.json()["response"]
print(paraphrase_answer)

The context does not specify this.


In [36]:
paraphrase_failure["context_sent_to_model"] = paraphrase_context
paraphrase_failure["model_answer"] = paraphrase_answer
paraphrase_failure["expected_answer"] = "The employee's manager."
paraphrase_failure["finding"] = (
    "Retrieval missed the evidence, causing abstention "
    "on a question answerable from the full policy."
)

Path("paraphrase_failure.json").write_text(
    json.dumps(paraphrase_failure, indent=2),
    encoding="utf-8"
)

print("Saved retrieval failure and model response.")

Saved retrieval failure and model response.


In [37]:
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "phi3:mini",
        "prompt": (
            "Answer only from the context below. "
            "If the answer is absent, reply exactly: "
            "'The context does not specify this.' "
            "Do not speculate or add explanations.\n\n"
            f"Context:\n{paraphrase_failure['expected_passage']}\n\n"
            f"Question: {paraphrase_failure['question']}"
        ),
        "stream": False,
        "options": {"temperature": 0, "num_predict": 150}
    },
    timeout=180
)

response.raise_for_status()
correct_context_answer = response.json()["response"]
print(correct_context_answer)

The employee's manager


In [38]:
paraphrase_failure["manual_evidence_control"] = {
    "context": paraphrase_failure["expected_passage"],
    "model_answer": correct_context_answer,
    "observation": "Correct answer when evidence was supplied manually."
}

Path("paraphrase_failure.json").write_text(
    json.dumps(paraphrase_failure, indent=2),
    encoding="utf-8"
)

print("Saved manual-evidence comparison.")

Saved manual-evidence comparison.
